# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ruchi74/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Section 1 — Research Findings and Methodology Questions

### Finding 1: Content decay may go unnoticed

FlyRank's project context describes how pages can lose search rankings and clicks over time, while teams may notice the decline too late.

**Methodology question:** Does the dataset include time-based information that can help identify declining content before the decline becomes severe?

### Finding 2: Fixed rules have limitations

FlyRank currently uses hand-written product flags, such as health scores and needs-attention tags. The project aims to investigate whether a trained model can provide useful prioritization beyond fixed rules.

**Methodology question:** Does the model perform better than a transparent rule-based baseline when both are evaluated on the same held-out clients, without using product flags as model features?

### Evidence limitation

These findings describe the FlyRank project context. They should not be presented as experimentally verified research-paper results. Quantitative conclusions require evaluation results from the actual research paper or capstone experiments.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load the FlyRank starter dataset
data_url = (
    "https://raw.githubusercontent.com/"
    "Ruchi74/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


Dataset loaded successfully!
Dataset shape: (30000, 44)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:

# STEP 3 — CHECK AVAILABLE TARGET LABELS

print("Dataset columns that may contain labels or outcomes:")
for col in df.columns:
    if any(word in col.lower() for word in ["label", "target", "declin", "trend"]):
        print("-", col)

print("\nTrend direction counts (inspection only):")
print(df["trend_direction"].value_counts(dropna=False))

print("\nTrend percentage summary (inspection only):")
print(df["trend_pct"].describe())

print("\nRecent and previous period columns:")
print([
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d"
])


Dataset columns that may contain labels or outcomes:
- trend_direction
- trend_pct

Trend direction counts (inspection only):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend percentage summary (inspection only):
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Recent and previous period columns:
['impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']


In [7]:

# STEP 4 — DEFINE A TARGET FROM RECENT VS PREVIOUS IMPRESSIONS

# Keep rows where both periods have usable impression counts
model_df = df[
    df["impressions_last_30d"].notna() &
    df["impressions_prev_30d"].notna() &
    (df["impressions_prev_30d"] >= 100)
].copy()

# Target: 1 means impressions declined by at least 20%;
# 0 means they did not decline by that amount.
model_df["target_decline"] = (
    model_df["impressions_last_30d"]
    <= 0.8 * model_df["impressions_prev_30d"]
).astype(int)

print("Rows available for modeling:", len(model_df))
print("\nTarget counts:")
print(model_df["target_decline"].value_counts().sort_index())

print("\nTarget proportions:")
print(
    model_df["target_decline"]
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
)

print("\nTarget definition:")
print("1 = impressions declined by at least 20%")
print("0 = impressions did not decline by at least 20%")


Rows available for modeling: 18010

Target counts:
target_decline
0     6909
1    11101
Name: count, dtype: int64

Target proportions:
target_decline
0    0.384
1    0.616
Name: proportion, dtype: float64

Target definition:
1 = impressions declined by at least 20%
0 = impressions did not decline by at least 20%


In [8]:


# STEP 5 — CLEAN FEATURE SPLIT

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

excluded_columns = [
    "content_id", "client_id", "target_decline",
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "ctr", "impressions_90d", "clicks_90d",
    "days_with_impressions", "impression_tier"
]

X = model_df.drop(columns=excluded_columns, errors="ignore")
y = model_df["target_decline"]

# Encode categorical columns
X = pd.get_dummies(X, drop_first=True, dtype=int)
X = X.replace([np.inf, -np.inf], np.nan)

# Split before imputing missing values
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Learn imputation values from training data only
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", X_train.shape[1])
print("Missing values in training:", X_train.isna().sum().sum())
print("Missing values in testing:", X_test.isna().sum().sum())


Training rows: 14408
Testing rows: 3602
Features: 47
Missing values in training: 0
Missing values in testing: 0


In [9]:


# STEP 6 — CHECK FOR TARGET LEAKAGE

print("Target definition:")
print("Decline = last 30-day impressions <= 80% of previous 30-day impressions")

print("\nExcluded target-related columns:")
print([
    "impressions_last_30d",
    "impressions_prev_30d",
    "trend_direction",
    "trend_pct"
])

print("\nAre target-related columns in model features?")
for col in [
    "impressions_last_30d",
    "impressions_prev_30d",
    "trend_direction",
    "trend_pct"
]:
    print(f"{col}: {col in X_train.columns}")

print("\nLeakage check complete.")



Target definition:
Decline = last 30-day impressions <= 80% of previous 30-day impressions

Excluded target-related columns:
['impressions_last_30d', 'impressions_prev_30d', 'trend_direction', 'trend_pct']

Are target-related columns in model features?
impressions_last_30d: False
impressions_prev_30d: False
trend_direction: False
trend_pct: False

Leakage check complete.


In [10]:


# STEP 7 — RETRAIN THE DECISION TREE

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("UPDATED DECISION TREE RESULTS")
print("-----------------------------")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1 Score:", round(f1_score(y_test, y_pred, zero_division=0), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


UPDATED DECISION TREE RESULTS
-----------------------------
Accuracy: 0.669
Precision: 0.743
Recall: 0.707
F1 Score: 0.725

Confusion Matrix:
[[ 840  542]
 [ 650 1570]]


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:




# STEP 8 — UPDATED BASELINE COMPARISON

baseline_pred = (
    (model_df.loc[X_test.index, "impressions_90d"] >= 500) &
    (model_df.loc[X_test.index, "avg_position"] > 0) &
    (model_df.loc[X_test.index, "avg_position"] <= 20) &
    (model_df.loc[X_test.index, "ctr"] < 0.5)
).astype(int)

comparison = pd.DataFrame({
    "Method": ["Week 4 Baseline", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, y_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, y_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0)
    ],
    "F1 Score": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0)
    ]
})

display(comparison.round(3))


,Method,Accuracy,Precision,Recall,F1 Score
0,Week 4 Baseline,0.526,0.639,0.533,0.581
1,Decision Tree,0.669,0.743,0.707,0.725


In [13]:
print("df available:", "df" in globals())
print("model_df available:", "model_df" in globals())
print("target available:", "target" in globals())

df available: True
model_df available: True
target available: False


In [14]:
# Create the target label from the trend direction
target = (model_df["trend_direction"] == "down").astype(int)

print("Target created successfully!")
print("\nTarget counts:")
print(target.value_counts())

print("\nDataset shape:", model_df.shape)
print("Target shape:", target.shape)

Target created successfully!

Target counts:
trend_direction
1    11090
0     6920
Name: count, dtype: int64

Dataset shape: (18010, 45)
Target shape: (18010,)


In [15]:
print("df shape:", df.shape)
print("model_df shape:", model_df.shape)

print("\nColumns in model_df:")
print(model_df.columns.tolist())

print("\nTrend direction counts in df:")
print(df["trend_direction"].value_counts(dropna=False))

df shape: (30000, 44)
model_df shape: (18010, 45)

Columns in model_df:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'target_decline']

Trend direction counts in df:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count,

In [16]:
# Recreate the Week 5 dataset from the original 30,000-row df
model_df = df.dropna(subset=["trend_direction"]).copy()

# Create the target: 1 = declining, 0 = not declining
target = (model_df["trend_direction"] == "down").astype(int)

print("Dataset shape:", model_df.shape)
print("Target shape:", target.shape)
print("\nTarget counts:")
print(target.value_counts())
print("\nUnique clients:", model_df["client_id"].nunique())

Dataset shape: (30000, 44)
Target shape: (30000,)

Target counts:
trend_direction
1    16262
0    13738
Name: count, dtype: int64

Unique clients: 32


In [17]:
from sklearn.model_selection import GroupShuffleSplit

# Columns that must not be used as model features
excluded_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target_decline",
]

# Prepare features and target
X = model_df.drop(
    columns=[c for c in excluded_cols if c in model_df.columns]
).copy()

# Replace infinite values with missing values
X = X.replace([float("inf"), float("-inf")], float("nan"))

# Encode categorical columns
X = pd.get_dummies(X, dummy_na=True)

# Split by client so clients do not overlap
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, target, groups=model_df["client_id"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = target.iloc[train_idx]
y_test = target.iloc[test_idx]

train_clients = set(model_df.iloc[train_idx]["client_id"])
test_clients = set(model_df.iloc[test_idx]["client_id"])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Overlapping clients:", len(train_clients & test_clients))
print("Feature count:", X.shape[1])

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Overlapping clients: 0
Feature count: 82


In [18]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# Convert all features to numeric and handle infinite values
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Learn imputation values from training data only
imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

# Train the model
model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
)

model.fit(X_train_imputed, y_train)

# Predict on unseen clients
y_pred = model.predict(X_test_imputed)

print("CLIENT-GROUPED MODEL RESULTS")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1 Score:", round(f1_score(y_test, y_pred, zero_division=0), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

CLIENT-GROUPED MODEL RESULTS
Accuracy: 0.666
Precision: 0.659
Recall: 0.717
F1 Score: 0.687

Confusion Matrix:
[[1845 1169]
 [ 891 2258]]


In [19]:
# Remove identifiers and columns that may reveal the target
excluded_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target_decline",
    "is_declining_label",

    # Recent and previous-period measurements used to derive trends
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",

    # Additional trend-related or derived features
    "ctr",
    "impressions_90d",
    "clicks_90d",
    "days_with_impressions",
    "impression_tier",
]

# Rebuild features, retaining only permitted columns
X = model_df.drop(
    columns=[c for c in excluded_cols if c in model_df.columns]
).copy()

X = pd.get_dummies(X, dummy_na=True)
X = X.replace([np.inf, -np.inf], np.nan)

# Repeat the client-grouped split
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, target, groups=model_df["client_id"])
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

# Fit imputer on training data only
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

imputer = SimpleImputer(strategy="median")

X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_imp, y_train)
y_pred = model.predict(X_test_imp)

train_clients = set(model_df.iloc[train_idx]["client_id"])
test_clients = set(model_df.iloc[test_idx]["client_id"])

print("FEATURES AFTER LEAKAGE EXCLUSIONS:", X.shape[1])
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Overlapping clients:", len(train_clients & test_clients))
print("\nCORRECTED CLIENT-GROUPED RESULTS")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1 Score:", round(f1_score(y_test, y_pred, zero_division=0), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

FEATURES AFTER LEAKAGE EXCLUSIONS: 67
Training clients: 25
Testing clients: 7
Overlapping clients: 0

CORRECTED CLIENT-GROUPED RESULTS
Accuracy: 0.516
Precision: 0.523
Recall: 0.599
F1 Score: 0.559

Confusion Matrix:
[[1296 1718]
 [1262 1887]]


In [20]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Simple baseline trained only on the training data
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_imp, y_train)

baseline_pred = baseline.predict(X_test_imp)

print("BASELINE RESULTS")
print("Accuracy:", round(accuracy_score(y_test, baseline_pred), 3))
print("Precision:", round(precision_score(y_test, baseline_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, baseline_pred, zero_division=0), 3))
print("F1 Score:", round(f1_score(y_test, baseline_pred, zero_division=0), 3))

BASELINE RESULTS
Accuracy: 0.511
Precision: 0.511
Recall: 1.0
F1 Score: 0.676


In [21]:
# Check for forbidden columns in the final feature set
forbidden_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target_decline",
    "is_declining_label",
]

print("LEAKAGE AUDIT")
print("-" * 40)

for col in forbidden_cols:
    print(f"{col}: {'FOUND - CHECK!' if col in X.columns else 'Not present'}")

print("\nClient overlap:", len(train_clients & test_clients))
print("Missing values in training:", int(X_train_imp.isna().sum().sum()))
print("Missing values in testing:", int(X_test_imp.isna().sum().sum()))
print("Training rows:", len(X_train_imp))
print("Testing rows:", len(X_test_imp))

LEAKAGE AUDIT
----------------------------------------
content_id: Not present
client_id: Not present
trend_direction: Not present
trend_pct: Not present
target_decline: Not present
is_declining_label: Not present

Client overlap: 0
Missing values in training: 0
Missing values in testing: 0
Training rows: 23837
Testing rows: 6163


## Section 2 — Before/After Validation Comparison

The initial Week 5 random-split model achieved 64.1% accuracy and an F1 score of 0.683.

For the Week 6 audit, I used a client-grouped split. The corrected model achieved 51.6% accuracy, 52.3% precision, 59.9% recall, and an F1 score of 0.559 on 6,163 test rows from seven clients not included in training.

The training set contained 25 clients, while the test set contained seven different clients, with zero client overlap. Potentially target-related features were excluded, and missing values were imputed using training data only.

The majority-class baseline achieved 51.1% accuracy and an F1 score of 0.676. Although the corrected model had slightly higher accuracy, its F1 score was lower than the baseline's. Therefore, the results do not demonstrate a clear overall improvement over the baseline.

The difference between the initial random-split and corrected client-grouped results suggests that the earlier evaluation may have overstated generalization performance. This is a directional finding and requires further validation.

## Section 3 — Leakage Audit

The final feature set was checked for direct identifiers and target-related columns.

- `content_id`: excluded.
- `client_id`: excluded from model features and used only for grouping.
- `trend_direction`: excluded.
- `trend_pct`: excluded.
- `target_decline` and `is_declining_label`: excluded.
- Client overlap between training and testing: zero.
- Missing values after imputation: zero in both sets.
- The imputer was fitted on training data only and then applied to the test data.

Potential leakage risks were reduced by excluding the identified columns and using client-grouped validation. However, this does not prove that all leakage has been eliminated. Other features may still be closely related to the target's definition or measurement period and require review.

### Additional Data Considerations

The starter dataset contains 30,000 content items from 32 pseudonymized clients, with trailing-90-day metrics. The dataset documentation warns that `avg_position = 0` represents missing data, rate columns are stored as percentage values multiplied by 100, and missingness can vary by content type. These data-quality issues should be considered when interpreting the results. The current audit does not establish that every possible leakage or data-quality risk has been eliminated.

## Section 4 — Claims Rewritten to Match the Evidence

**Claim 1: Original claim**

The model accurately predicts declining content.

**Rewritten claim**

On the client-grouped test set, the corrected model achieved 51.6% accuracy and an F1 score of 0.559. Its F1 score was lower than the majority-class baseline's 0.676, so the current results do not establish reliable predictive performance.

**Claim 2: Original claim**

The model generalizes well to new clients.

**Rewritten claim**

The model was evaluated on seven held-out clients with zero client overlap between training and testing. The results provide an initial assessment of generalization to unseen clients, but further validation is needed.

**Claim 3: Original claim**

The model is ready for production use.

**Rewritten claim**

The current model may be explored as a decision-support prototype. More validation, feature review, and testing on additional clients are needed before considering production use.

## Section 5 — Self-Check

- [x] Used a client-grouped split for validation.
- [x] Confirmed zero client overlap between training and testing.
- [x] Excluded direct identifiers and identified target-related columns.
- [x] Fitted the imputer on training data only.
- [x] Compared the model against a majority-class baseline.
- [x] Reported the results honestly without unsupported claims.
- [ ] Reviewed two actual findings from the FlyRank research paper and added methodology questions.
- [ ] Reran all notebook cells from top to bottom.
- [ ] Saved and committed the completed notebook to GitHub.

### Final Conclusion

The corrected model achieved 51.6% accuracy and an F1 score of 0.559 on the client-grouped test set. The majority-class baseline achieved 51.1% accuracy and an F1 score of 0.676. Although the model had slightly higher accuracy, its F1 score was lower than the baseline's.

The validation results do not establish an overall improvement over the baseline. The client-grouped split provides a more realistic assessment of performance on unseen clients. Further validation and feature review are needed before considering production use. The current model should be treated as an experimental decision-support prototype.